# Hyper Fingerprints — Quickstart

This notebook shows how to use **hyper_fingerprints** to encode molecules into
fixed-size hypervectors using Holographic Reduced Representations (HRR) with
message passing.

The only dependencies are **numpy** and **rdkit**.

## 1. Basic encoding

In [ ]:
from hyper_fingerprints import Encoder

enc = Encoder()
enc

: 

Encode a single molecule from a SMILES string:

In [ ]:
fp = enc.encode("CCO")  # ethanol
print(f"Shape: {fp.shape}")  # (1, 256)
print(f"dtype: {fp.dtype}")
fp

Encode a batch of molecules in one call:

In [ ]:
smiles = ["CCO", "c1ccccc1", "CC(=O)O", "CC(=O)Nc1ccc(O)cc1"]
fps = enc.encode(smiles)
print(f"Shape: {fps.shape}")  # (4, 256)

You can also pass RDKit `Mol` objects directly:

In [ ]:
from rdkit import Chem

mol = Chem.MolFromSmiles("c1ccccc1")
fp = enc.encode(mol)
print(f"Shape: {fp.shape}")  # (1, 256)

## 2. Configuring the encoder

The main parameters are:

| Parameter | Default | Description |
|-----------|---------|-------------|
| `dimension` | 256 | Hypervector size |
| `depth` | 3 | Message-passing layers (controls how much structural context is captured) |
| `seed` | None | Fix the random codebook for reproducibility |
| `normalize` | False | L2-normalize after each message-passing layer |
| `atom_types` | 9 common types | Which elements are supported |

In [ ]:
enc_large = Encoder(dimension=1024, depth=5, seed=42)

fp = enc_large.encode("CCO")
print(f"Shape: {fp.shape}")  # (1, 1024)

Using a fixed `seed` makes results deterministic:

In [ ]:
import numpy as np

enc_a = Encoder(seed=123)
enc_b = Encoder(seed=123)

fp_a = enc_a.encode("c1ccccc1")
fp_b = enc_b.encode("c1ccccc1")

print(f"Identical: {np.allclose(fp_a, fp_b)}")

## 3. Joint fingerprints

`encode_joint()` returns the concatenation of two views:

- **Order-0** (first half) — atom identity only, no structural context
- **Order-N** (second half) — full message-passing, same as `encode()`

This is useful when you want both local and structural information.

In [ ]:
enc = Encoder(dimension=256, seed=42)

joint = enc.encode_joint("CCO")
print(f"Joint shape: {joint.shape}")  # (1, 512) = 2 * 256

order_0 = joint[:, :enc.dimension]
order_n = joint[:, enc.dimension:]

# The second half matches encode()
fp = enc.encode("CCO")
print(f"Order-N matches encode(): {np.allclose(order_n, fp)}")

## 4. Similarity search

Since fingerprints are real-valued vectors, cosine similarity is a natural
way to compare molecules.

In [ ]:
enc = Encoder(dimension=512, seed=42)

molecules = {
    "ethanol":       "CCO",
    "methanol":      "CO",
    "acetic acid":   "CC(=O)O",
    "benzene":       "c1ccccc1",
    "toluene":       "Cc1ccccc1",
    "phenol":        "Oc1ccccc1",
    "acetaminophen": "CC(=O)Nc1ccc(O)cc1",
}

names = list(molecules.keys())
smiles = list(molecules.values())
fps = enc.encode(smiles)  # (7, 512)

In [ ]:
def cosine_similarity_matrix(X):
    """Compute pairwise cosine similarity."""
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X_normed = X / norms
    return X_normed @ X_normed.T

sim = cosine_similarity_matrix(fps)

# Print similarity matrix
header = f"{'':>15s}" + "".join(f"{n:>15s}" for n in names)
print(header)
for i, name in enumerate(names):
    row = f"{name:>15s}" + "".join(f"{sim[i, j]:>15.3f}" for j in range(len(names)))
    print(row)

Find the most similar molecule to a query:

In [ ]:
query = enc.encode("Cc1ccc(O)cc1")  # p-cresol (toluene + phenol)

# Cosine similarity against all molecules
query_norm = query / np.linalg.norm(query)
fps_norm = fps / np.linalg.norm(fps, axis=1, keepdims=True)
similarities = (query_norm @ fps_norm.T).squeeze()

ranking = np.argsort(-similarities)
print("Most similar to p-cresol (Cc1ccc(O)cc1):")
for idx in ranking:
    print(f"  {names[idx]:>15s}  ({smiles[idx]:>20s})  sim = {similarities[idx]:.4f}")

## 5. Custom atom types

By default the encoder supports: `Br, C, Cl, F, I, N, O, P, S`.

If your molecules only contain a subset, you can restrict the vocabulary
(this makes the codebook smaller):

In [ ]:
from hyper_fingerprints import DEFAULT_ATOM_TYPES

print(f"Default atom types: {DEFAULT_ATOM_TYPES}")

# Organic-only encoder
enc_organic = Encoder(atom_types=["C", "N", "O"], seed=42)
print(f"Feature bins: {enc_organic.feature_bins}")  # [3, 6, 3, 4, 2]

fp = enc_organic.encode("CCO")
print(f"Shape: {fp.shape}")

Encoding a molecule with an unsupported atom type raises an error:

In [ ]:
try:
    enc_organic.encode("c1ccc(F)cc1")  # fluorine not in [C, N, O]
except ValueError as e:
    print(f"Error: {e}")

## 6. Saving and loading an encoder

You can save a configured encoder (including its codebook) to disk and load it
back later. This is useful for sharing a fixed fingerprint scheme or deploying
in production without needing to keep track of the seed.

In [ ]:
enc = Encoder(dimension=512, seed=42)
fp_before = enc.encode("CCO")

# Save to disk
enc.save("my_encoder.npz")

# Load in a new session (or a different machine)
loaded = Encoder.load("my_encoder.npz")

fp_after = loaded.encode("CCO")
print(f"Fingerprints identical: {np.allclose(fp_before, fp_after)}")
print(f"Config: dimension={loaded.dimension}, depth={loaded.depth}, seed={loaded.seed}")

## 7. Using with scikit-learn

Since `encode()` returns plain numpy arrays, integration with scikit-learn
(or any other ML library) is straightforward.

In [ ]:
# Example: cluster molecules by their fingerprints
enc = Encoder(dimension=512, seed=42)

smiles_set = [
    # alcohols
    "CO", "CCO", "CCCO", "CCCCO",
    # aromatics
    "c1ccccc1", "Cc1ccccc1", "c1ccc(O)cc1", "c1ccc(N)cc1",
    # acids
    "CC(=O)O", "CCC(=O)O", "CCCC(=O)O", "OC(=O)c1ccccc1",
]

X = enc.encode(smiles_set)  # (12, 512)
print(f"Feature matrix: {X.shape}")
print("\nReady for sklearn — e.g. KMeans, PCA, classifiers, etc.")